In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

from model_ranking import load_transfer_metric_results, correlation_table

In [9]:
transfer_metric_abbrevs = {
    "LEEP": "LEEP",
    "NCTI": "NCTI",
    "LogME": "LogME",
    "Hscore": "Hscore",
    "GBC": "GBC",
    "Regularized_Hscore": "RegHscore",
    "Gaussian_LEEP": "NLEEP",
}

In [11]:
base_path = "/g/kreshuk/talks/sampled_features/classification/mitochondria/transfer_metric_results/fullset_results"
result_paths = Path(base_path).glob("*.json")

# Collect all dataframes
all_dfs = []

for result_path in result_paths:
    results = load_transfer_metric_results(result_path)
    correlation_scores = results["correlation_scores"]
    idx = 0
    per_target_KT = np.array(correlation_scores["KT"])
    per_target_SP = np.array(correlation_scores["SP"])
    per_target_PE = np.array(correlation_scores["PE"])
    df = correlation_table(per_target_KT, per_target_SP, per_target_PE, targets = ["EPFL", "Hmito", "Rmito"])
    transfer_metric_name = '_'.join(result_path.stem.split("_")[1:])
    transfer_metric_abbrev = transfer_metric_abbrevs[transfer_metric_name]
    
    # Remove the Task index level and add transfer_metric index
    df = df.droplevel("Task")
    df["transfer_metric"] = transfer_metric_abbrev
    df = df.set_index("transfer_metric", append=True)
    
    all_dfs.append(df)

# Combine all dataframes
combined_df = pd.concat(all_dfs)

# Reorder the index levels to have transfer_metric first
combined_df = combined_df.reorder_levels(["transfer_metric", "targets"])

print("Combined correlation table for all transfer metrics:")
print(combined_df)

Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/classification/mitochondria/transfer_metric_results/fullset_results/fullset_NCTI.json
Experiment: fullset_NCTI
Targets: 3 (EPFL, Hmito, Rmito)
Source models: 11
Total transfers: 33
Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/classification/mitochondria/transfer_metric_results/fullset_results/fullset_GBC.json
Experiment: fullset_GBC
Targets: 3 (EPFL, Hmito, Rmito)
Source models: 11
Total transfers: 33
Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/classification/mitochondria/transfer_metric_results/fullset_results/fullset_LEEP.json
Experiment: fullset_LEEP
Targets: 3 (EPFL, Hmito, Rmito)
Source models: 11
Total transfers: 33
Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/classification/mitochondria/transfer_metric_results/fullset_results/fullset_Gaussian_LEEP.json
Experiment: fullset_Gaussian_LEEP
Targets: 3 (EPFL, Hmito, Rmito)
Source models:

In [13]:
# Display the shape and structure of the combined table
print(f"Combined table shape: {combined_df.shape}")
print(f"Index levels: {combined_df.index.names}")
print(f"Transfer metrics: {combined_df.index.get_level_values('transfer_metric').unique().tolist()}")
print(f"Targets: {combined_df.index.get_level_values('targets').unique().tolist()}")
print()

# Show a sample of the data - first few rows for each transfer metric
print("Sample of combined correlation table:")
for metric in combined_df.index.get_level_values('transfer_metric').unique()[:3]:
    print(f"\n{metric}:")
    print(combined_df.loc[metric])

# You can also access specific combinations like this:
print("\nExample: LEEP results for EPFL:")
print(combined_df.loc[('LEEP', 'EPFL')])

# Or get all results for a specific target across all transfer metrics:
print("\nAll transfer metrics for Hmito target:")
print(combined_df.xs('Hmito', level='targets'))

Combined table shape: (21, 6)
Index levels: ['transfer_metric', 'targets']
Transfer metrics: ['NCTI', 'GBC', 'LEEP', 'NLEEP', 'Hscore', 'RegHscore', 'LogME']
Targets: ['EPFL', 'Hmito', 'Rmito']

Sample of combined correlation table:

NCTI:
           kt  kt pval  s rho  s rho pval    pr  pr pval
targets                                                 
EPFL     0.88      0.0   0.65        0.03  0.45     0.08
Hmito    0.83      0.0   0.80        0.00  0.60     0.01
Rmito    0.87      0.0   0.85        0.00  0.75     0.01

GBC:
           kt  kt pval  s rho  s rho pval    pr  pr pval
targets                                                 
EPFL     0.47     0.15   0.21        0.54  0.24     0.39
Hmito    0.79     0.00   0.83        0.00  0.67     0.00
Rmito    0.66     0.03   0.69        0.02  0.49     0.04

LEEP:
           kt  kt pval  s rho  s rho pval    pr  pr pval
targets                                                 
EPFL     0.97      0.0   0.95         0.0  0.85      0.0
Hmito 

In [16]:
print("\nAll transfer metrics for Hmito target:")
print(combined_df.xs('Hmito', level='targets'))


All transfer metrics for Hmito target:
                   kt  kt pval  s rho  s rho pval    pr  pr pval
transfer_metric                                                 
NCTI             0.83      0.0   0.80         0.0  0.60     0.01
GBC              0.79      0.0   0.83         0.0  0.67     0.00
LEEP             0.98      0.0   0.99         0.0  0.96     0.00
NLEEP            0.99      0.0   0.99         0.0  0.96     0.00
Hscore           0.96      0.0   0.98         0.0  0.93     0.00
RegHscore        0.96      0.0   0.98         0.0  0.93     0.00
LogME            0.97      0.0   0.98         0.0  0.93     0.00


In [14]:
print("\nAll transfer metrics for EPFL target:")
print(combined_df.xs('EPFL', level='targets'))


All transfer metrics for EPFL target:
                   kt  kt pval  s rho  s rho pval    pr  pr pval
transfer_metric                                                 
NCTI             0.88     0.00   0.65        0.03  0.45     0.08
GBC              0.47     0.15   0.21        0.54  0.24     0.39
LEEP             0.97     0.00   0.95        0.00  0.85     0.00
NLEEP            0.80     0.00   0.73        0.01  0.56     0.01
Hscore           0.82     0.00   0.72        0.01  0.53     0.03
RegHscore        0.83     0.00   0.79        0.01  0.60     0.01
LogME            0.85     0.00   0.75        0.02  0.53     0.03


In [15]:
print("\nAll transfer metrics for Rmito target:")
print(combined_df.xs('Rmito', level='targets'))


All transfer metrics for Rmito target:
                   kt  kt pval  s rho  s rho pval    pr  pr pval
transfer_metric                                                 
NCTI             0.87     0.00   0.85        0.00  0.75     0.01
GBC              0.66     0.03   0.69        0.02  0.49     0.04
LEEP             0.98     0.00   0.98        0.00  0.93     0.00
NLEEP            0.92     0.00   0.87        0.00  0.75     0.00
Hscore           0.83     0.00   0.77        0.01  0.64     0.01
RegHscore        0.83     0.00   0.76        0.02  0.64     0.01
LogME            0.88     0.00   0.85        0.00  0.71     0.00
